# 依存距離と構文木の深さで文の複雑さを測る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_parsing/dep_distance.ipynb)

NLP クックブック 17-11 ／ 構文解析

文の構造的な複雑さを数値化したい（可読性の評価、学習者作文の分析、言語間の比較）。

In [1]:
!pip install -q ginza ja_ginza

In [2]:
import spacy

nlp = spacy.load("ja_ginza")

def depth(token):
    return 0 if token.head.i == token.i else 1 + depth(token.head)

def complexity(text):
    rows = []
    for sent in nlp(text).sents:
        toks = [t for t in sent if t.pos_ != "PUNCT"]
        dists = [abs(t.i - t.head.i) for t in toks if t.head.i != t.i]
        rows.append({"文": sent.text[:22], "語数": len(toks),
                     "平均依存距離": round(sum(dists) / max(1, len(dists)), 2),
                     "最大依存距離": max(dists, default=0),
                     "木の深さ": max(depth(t) for t in toks)})
    return rows

import pandas as pd

text = ("猫が寝ている。"
        "太郎が昨日図書館で借りた本を花子が読んでいる。"
        "私が先週友人から勧められて書店で買った小説を弟が勝手に持ち出して学校で友達に貸してしまった。")
pd.DataFrame(complexity(text))

,文,語数,平均依存距離,最大依存距離,木の深さ
0,猫が寝ている。,5,1.25,2,2
1,太郎が昨日図書館で借りた本を花子が読んでいる,14,1.92,5,4
2,私が先週友人から勧められて書店で買った小説を,27,2.19,6,6


- 平均依存距離 (MDD) は、係り元と係り先が離れているほど大きくなり、ワーキングメモリの負荷の指標とされる。
- 日本語は主辞後置型なので、長い連体修飾節があると依存距離が大きくなる。